In [1]:
pip install pandas numpy matplotlib seaborn jupyter

  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached notebook-7.6.3-py3-none-any.whl.metadata (10 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached ipywidgets-8.1.9-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab-4.6.4-py3-none-any.whl.metadata (16 kB)
  Using cached widgetsnbextension-4.0.16-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl.metadata (20 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_builder-1.2.3-py3-none-any.whl.metadata (7.8 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.21.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached beautifulsoup4

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\bhask\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\jupyterlab\\galata\\@jupyterlab\\galata-extension\\static\\vendors-node_modules_fontsource-variable_noto-sans-sc_index_css.386da7677f8b1f52.js'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\bhask\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import os

input_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed\global_startup_cleaned.csv"

df_clean = pd.read_csv(input_path)

sql_columns = [
    "permalink",
    "name",
    "market",
    "category_list",
    "country_code",
    "state_code",
    "region",
    "city",
    "status",
    "founded_year",
    "funding_rounds",
    "funding_total_numeric",
    "first_funding_at_parsed",
    "last_funding_at_parsed",
    "suspicious_funding_date"
]

df_sql = df_clean[sql_columns].copy()

# Convert numeric columns safely
for col in ["founded_year", "funding_rounds", "funding_total_numeric"]:
    df_sql[col] = pd.to_numeric(df_sql[col], errors="coerce")

# Convert Boolean flags to 0 and 1
df_sql["suspicious_funding_date"] = (
    df_sql["suspicious_funding_date"]
    .map({True: 1, False: 0, "True": 1, "False": 0})
)

# Convert date columns to proper dates
date_columns = [
    "first_funding_at_parsed",
    "last_funding_at_parsed"
]

for col in date_columns:
    df_sql[col] = pd.to_datetime(df_sql[col], errors="coerce").dt.strftime("%Y-%m-%d")

# Export without writing NaN or NaT as text
output_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed"

os.makedirs(output_path, exist_ok=True)

df_sql.to_csv(
    os.path.join(output_path, "global_startup_sql.csv"),
    index=False,
    na_rep=""
)

print("Rows:", len(df_sql))
print("Columns:", len(df_sql.columns))
print("Missing founded_year:", df_sql["founded_year"].isna().sum())
print("Missing funding_total_numeric:", df_sql["funding_total_numeric"].isna().sum())
print("CSV exported successfully!")

Rows: 49438
Columns: 15
Missing founded_year: 10956
Missing funding_total_numeric: 8531
CSV exported successfully!


In [6]:
empty_rows  = df.isna().all(axis=1).sum()
non_empty_rows = df.notna().any(axis=1).sum()

print(f"Completely empty rows: {empty_rows:,}")
print(f"Non-empty rows: {non_empty_rows:,}")
print(f"Total rows: {len(df):,}")

Completely empty rows: 4,856
Non-empty rows: 49,438
Total rows: 54,294


In [8]:
missing_values = df.isna().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_summary = pd.DataFrame({"Missing Values": missing_values,"Missing Percentage": missing_percentage.round(2)})
missing_summary = missing_summary[missing_summary["Missing Values"] > 0].sort_values(by = "Missing Values", ascending=False)

display(missing_summary)

,Missing Values,Missing Percentage
state_code,24133,44.45
founded_quarter,15812,29.12
founded_month,15812,29.12
founded_year,15812,29.12
founded_at,15740,28.99
city,10972,20.21
region,10129,18.66
country_code,10129,18.66
market,8824,16.25
category_list,8817,16.24


In [9]:
duplicate_rows = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows},")

Exact duplicate rows: 4855,


In [11]:
non_empty_df = df.dropna(how="all")

duplicate_permalink_count = non_empty_df[
    non_empty_df["permalink"].notna()
]["permalink"].duplicated().sum()

print(
    "Repeated company identifiers after the first occurrence:",
    duplicate_permalink_count
)

Repeated company identifiers after the first occurrence: 2


In [12]:
duplicate_permalink_mask = (
    non_empty_df["permalink"].notna()
    & non_empty_df["permalink"].duplicated(keep = False)
)

duplicate_permalink_records = non_empty_df[
    duplicate_permalink_mask].sort_values("permalink")

display(duplicate_permalink_records)

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
33939,/organization/prysm,Prysm,http://www.prysm.com/,NaN,NaN,"29,30,80,123",operating,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,2014-11-25,2014-11-25,0.0,2750000.0,0.0,0.0,0.0,0.0,0.0,0.0,290330123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33940,/organization/prysm,Prysm,http://www.prysm.com,|Displays|Hardware + Software|,Displays,"29,30,80,123",operating,USA,CA,SF Bay Area,San Jose,3.0,2005-01-01,2005-01,2005-Q1,2005.0,2010-01-05,2011-10-27,0.0,2750000.0,0.0,0.0,0.0,0.0,0.0,0.0,290330123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44033,/organization/treasure-valley-urology-services,Treasure Valley Urology Services,NaN,|Biotechnology|,Biotechnology,"3,32,194",operating,USA,TX,Austin,Austin,4.0,2004-01-01,2004-01,2004-Q1,2004.0,2009-09-09,2014-05-05,45321.0,286873.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44034,/organization/treasure-valley-urology-services,Treasure Valley Urology Services,NaN,NaN,NaN,"3,32,194",operating,USA,TX,Austin,Austin,1.0,2004-01-01,2004-01,2004-Q1,2004.0,2014-07-18,2014-07-18,45321.0,286873.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
df_clean = df.dropna(how="all").copy()

print("Original dataset shape:",df.shape)
print("Cleaned dataset shape:",df_clean.shape)

print("Rows removed:",len(df) - len(df_clean))

Original dataset shape: (54294, 39)
Cleaned dataset shape: (49438, 39)
Rows removed: 4856


In [15]:
df_clean.columns = df_clean.columns.str.strip()
print(df_clean.columns.tolist())

['permalink', 'name', 'homepage_url', 'category_list', 'market', 'funding_total_usd', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']


In [17]:
missing_values = df_clean.isna().sum()

missing_percentage = ( 
    missing_values / len(df_clean)
) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage.round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values(
    by="Missing Percentage",
    ascending=False
)

In [18]:
exact_duplicates = df_clean.duplicated().sum()
print(f"Exact duplicate records: {exact_duplicates:,}")

Exact duplicate records: 0


In [19]:
print(df_clean.dtypes)

permalink                   str
name                        str
homepage_url                str
category_list               str
market                      str
funding_total_usd           str
status                      str
country_code                str
state_code                  str
region                      str
city                        str
funding_rounds          float64
founded_at                  str
founded_month               str
founded_quarter             str
founded_year            float64
first_funding_at            str
last_funding_at             str
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A 

In [20]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 49438 entries, 0 to 49437
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   permalink             49438 non-null  str    
 1   name                  49437 non-null  str    
 2   homepage_url          45989 non-null  str    
 3   category_list         45477 non-null  str    
 4   market                45470 non-null  str    
 5   funding_total_usd     49438 non-null  str    
 6   status                48124 non-null  str    
 7   country_code          44165 non-null  str    
 8   state_code            30161 non-null  str    
 9   region                44165 non-null  str    
 10  city                  43322 non-null  str    
 11  funding_rounds        49438 non-null  float64
 12  founded_at            38554 non-null  str    
 13  founded_month         38482 non-null  str    
 14  founded_quarter       38482 non-null  str    
 15  founded_year          38482 no

In [21]:
display(df_clean["funding_total_usd"].head(20))

0      17,50,000 
1      40,00,000 
2         40,000 
3      15,00,000 
4         60,000 
5      70,00,000 
6      49,12,393 
7      20,00,000 
8            -   
9         41,250 
10        40,000 
11           -   
12           -   
13     17,50,000 
14           -   
15     44,00,000 
16     20,50,000 
17        40,000 
18      5,00,000 
19           -   
Name: funding_total_usd, dtype: str

In [22]:
display(
    df_clean["funding_total_usd"]
    .dropna()
    .drop_duplicates()
    .head(30)
)

0        17,50,000 
1        40,00,000 
2           40,000 
3        15,00,000 
4           60,000 
5        70,00,000 
6        49,12,393 
7        20,00,000 
8              -   
9           41,250 
15       44,00,000 
16       20,50,000 
18        5,00,000 
20       25,35,000 
21       49,62,651 
22       40,59,079 
23     1,00,00,000 
24       30,00,000 
25       45,00,000 
26        4,20,000 
27       12,50,000 
28     3,50,00,000 
30          50,000 
31       16,00,000 
32        1,00,000 
35          77,500 
37     2,00,00,000 
38       18,00,000 
43        2,70,000 
44        7,94,000 
Name: funding_total_usd, dtype: str

In [23]:
dash_count = (
    df_clean["funding_total_usd"]
    .astype("string")
    .str.strip()
    .eq("-")
    .sum()
)

print(f"Funding amounts represented by '-':{dash_count}")

Funding amounts represented by '-':8531


In [24]:
funding_values = (
    df_clean["funding_total_usd"]
    .astype("string")
    .str.strip()
)

print("Smallest values alphabetically:")
display(funding_values.sort_values().head(20))

print("Largest values alphabetically:")
display(funding_values.sort_values().tail(20))

Smallest values alphabetically:


8     -
11    -
12    -
14    -
19    -
29    -
33    -
36    -
39    -
40    -
41    -
42    -
46    -
50    -
52    -
61    -
66    -
69    -
70    -
71    -
Name: funding_total_usd, dtype: string

Largest values alphabetically:


8870     99,15,000
4079     99,15,755
25092    99,18,168
936         99,234
32886    99,29,825
14676       99,310
40512       99,362
12374    99,50,000
19369    99,50,000
10799    99,50,002
37152    99,52,199
14278    99,57,650
14696       99,627
43577    99,80,000
33171    99,81,363
34397       99,875
9525     99,90,000
30559    99,90,000
45993    99,99,997
4520     99,99,999
Name: funding_total_usd, dtype: string

In [25]:
unusual_funding_values = funding_values[
    ~funding_values.str.fullmatch(
        r"[\d,.\-\s]+",
        na=False
    )
]

print(
    "Funding values containing unexpected characters:",
    len(unusual_funding_values)
)

display(unusual_funding_values.drop_duplicates().head(50))

Funding values containing unexpected characters: 0


Series([], Name: funding_total_usd, dtype: string)

In [26]:
funding_columns = [
    "seed",
    "venture",
    "equity_crowdfunding",
    "undisclosed",
    "convertible_note",
    "debt_financing",
    "angel",
    "grant",
    "private_equity",
    "post_ipo_equity",
    "post_ipo_debt",
    "secondary_market",
    "product_crowdfunding"
]

df_clean["funding_components_sum"] = df_clean[
    funding_columns
].sum(axis=1)

print("Sample funding categories and totals:")

display(
    df_clean[
        ["name", "funding_total_usd", "funding_components_sum"]
        + funding_columns[:5]
    ].head(15)
)

Sample funding categories and totals:


,name,funding_total_usd,funding_components_sum,seed,venture,equity_crowdfunding,undisclosed,convertible_note
0,#waywire,"17,50,000",1750000.0,1750000.0,0.0,0.0,0.0,0.0
1,&TV Communications,"40,00,000",4000000.0,0.0,4000000.0,0.0,0.0,0.0
2,'Rock' Your Paper,"40,000",40000.0,40000.0,0.0,0.0,0.0,0.0
3,(In)Touch Network,"15,00,000",1500000.0,1500000.0,0.0,0.0,0.0,0.0
4,-R- Ranch and Mine,"60,000",60000.0,0.0,0.0,60000.0,0.0,0.0
5,.Club Domains,"70,00,000",7000000.0,0.0,7000000.0,0.0,0.0,0.0
6,.Fox Networks,"49,12,393",4912393.0,0.0,0.0,0.0,4912393.0,0.0
7,0-6.com,"20,00,000",2000000.0,0.0,2000000.0,0.0,0.0,0.0
8,004 Technologies,-,0.0,0.0,0.0,0.0,0.0,0.0
9,01Games Technology,"41,250",41250.0,41250.0,0.0,0.0,0.0,0.0


In [27]:
print(
    "Fractional funding-round counts:",
    (
        df_clean["funding_rounds"].dropna()
        % 1 != 0
    ).sum()
)

print(
    "Negative funding-round counts:",
    (df_clean["funding_rounds"] < 0).sum()
)

print(
    "Zero funding-round counts:",
    (df_clean["funding_rounds"] == 0).sum()
)

Fractional funding-round counts: 0
Negative funding-round counts: 0
Zero funding-round counts: 0


In [28]:
# Create a cleaned version of the funding column
funding_text = (
    df_clean["funding_total_usd"]
    .astype("string")
    .str.strip()
)

# Treat dashes as missing values
funding_text = funding_text.replace("-", pd.NA)

# Remove comma separators
funding_text = funding_text.str.replace(",", "", regex=False)

# Convert to numeric
df_clean["funding_total_numeric"] = pd.to_numeric(
    funding_text,
    errors="coerce"
)

# Validate the conversion
print("Original funding column:")
display(df_clean["funding_total_usd"].head(10))

print("\nConverted funding column:")
display(df_clean["funding_total_numeric"].head(10))

print(
    "\nMissing numeric funding amounts:",
    df_clean["funding_total_numeric"].isna().sum()
)

print(
    "Numeric funding data type:",
    df_clean["funding_total_numeric"].dtype
)

Original funding column:


0     17,50,000 
1     40,00,000 
2        40,000 
3     15,00,000 
4        60,000 
5     70,00,000 
6     49,12,393 
7     20,00,000 
8           -   
9        41,250 
Name: funding_total_usd, dtype: str


Converted funding column:


0    1750000
1    4000000
2      40000
3    1500000
4      60000
5    7000000
6    4912393
7    2000000
8       <NA>
9      41250
Name: funding_total_numeric, dtype: Int64


Missing numeric funding amounts: 8531
Numeric funding data type: Int64


In [29]:
original_dash_count = (
    df_clean["funding_total_usd"]
    .astype("string")
    .str.strip()
    .eq("-")
    .sum()
)

converted_missing_count = (
    df_clean["funding_total_numeric"]
    .isna()
    .sum()
)

print("Original dash values:", original_dash_count)
print("Missing values after conversion:", converted_missing_count)

print(
    "Unexpected conversion failures:",
    converted_missing_count - original_dash_count
)

Original dash values: 8531
Missing values after conversion: 8531
Unexpected conversion failures: 0


In [30]:
df_clean["funding_components_sum"] = (
    df_clean[funding_columns].sum(axis=1)
)

known_total_mask = df_clean["funding_total_numeric"].notna()

df_clean["funding_difference"] = (
    df_clean["funding_total_numeric"]
    - df_clean["funding_components_sum"]
)

matching_totals = (
    df_clean.loc[known_total_mask, "funding_difference"] == 0
)

print("Companies with known funding totals:", known_total_mask.sum())
print("Exact matches:", matching_totals.sum())
print("Mismatches:", (~matching_totals).sum())

print(
    "Percentage of known totals that match:",
    round(matching_totals.mean() * 100, 2),
    "%"
)

Companies with known funding totals: 40907
Exact matches: 40907
Mismatches: 0
Percentage of known totals that match: 100.0 %


In [31]:
df_clean["funding_rounds"] = (
    df_clean["funding_rounds"].astype("int64")
)

print("Data type:", df_clean["funding_rounds"].dtype)

display(
    df_clean["funding_rounds"]
    .value_counts()
    .sort_index()
    .head(15)
)

Data type: int64


funding_rounds
1     32039
2      9219
3      4026
4      1997
5      1001
6       560
7       252
8       152
9        84
10       43
11       35
12       12
13        8
14        4
15        4
Name: count, dtype: int64

In [32]:
date_columns = [
    "founded_at",
    "first_funding_at",
    "last_funding_at"
]

for column in date_columns:
    print(f"\n{column}")
    display(
        df_clean[column]
        .dropna()
        .drop_duplicates()
        .head(10)
    )


founded_at


0     2012-06-01
2     2012-10-26
3     2011-04-01
4     2014-01-01
5     2011-10-10
7     2007-01-01
8     2010-01-01
10    2012-01-01
12    1986-01-01
13    1984-01-01
Name: founded_at, dtype: str


first_funding_at


0    2012-06-30
1    2010-06-04
2    2012-08-09
3    2011-04-01
4    2014-08-17
5    2013-05-31
6    2007-01-16
7    2008-03-19
8    2014-07-24
9    2014-07-01
Name: first_funding_at, dtype: str


last_funding_at


0    2012-06-30
1    2010-09-23
2    2012-08-09
3    2011-04-01
4    2014-09-26
5    2013-05-31
6    2007-01-16
7    2008-03-19
8    2014-07-24
9    2014-07-01
Name: last_funding_at, dtype: str

In [33]:
for column in date_columns:
    df_clean[f"{column}_parsed"] = pd.to_datetime(
        df_clean[column],
        errors="coerce"
    )

    print(f"\n{column}")
    print(
        "Original non-missing values:",
        df_clean[column].notna().sum()
    )
    print(
        "Parsed non-missing values:",
        df_clean[f"{column}_parsed"].notna().sum()
    )
    print(
        "Parsed missing values:",
        df_clean[f"{column}_parsed"].isna().sum()
    )


founded_at
Original non-missing values: 38554
Parsed non-missing values: 38554
Parsed missing values: 10884

first_funding_at
Original non-missing values: 49438
Parsed non-missing values: 49438
Parsed missing values: 0

last_funding_at
Original non-missing values: 49438
Parsed non-missing values: 49438
Parsed missing values: 0


In [34]:
print("Earliest founding year:", df_clean["founded_year"].min())
print("Latest founding year:", df_clean["founded_year"].max())

print("\nFounding year frequency:")
display(
    df_clean["founded_year"]
    .value_counts()
    .sort_index()
    .head(20)
)

Earliest founding year: 1902.0
Latest founding year: 2014.0

Founding year frequency:


founded_year
1902.0    2
1903.0    1
1905.0    1
1906.0    5
1907.0    1
1908.0    1
1910.0    2
1911.0    2
1912.0    6
1913.0    2
1914.0    2
1917.0    1
1918.0    1
1919.0    3
1920.0    1
1921.0    2
1922.0    4
1923.0    2
1924.0    6
1925.0    1
Name: count, dtype: int64

In [35]:
founded_date_year = df_clean["founded_at_parsed"].dt.year

comparable_rows = (
    df_clean["founded_year"].notna()
    & founded_date_year.notna()
)

year_mismatches = (
    df_clean.loc[comparable_rows, "founded_year"]
    != founded_date_year.loc[comparable_rows]
)

print("Rows with both founding date and year:", comparable_rows.sum())
print("Founding-year mismatches:", year_mismatches.sum())

Rows with both founding date and year: 38482
Founding-year mismatches: 0


In [36]:
funding_date_mask = (
    df_clean["first_funding_at_parsed"].notna()
    & df_clean["last_funding_at_parsed"].notna()
)

funding_date_errors = (
    df_clean.loc[funding_date_mask, "first_funding_at_parsed"]
    > df_clean.loc[funding_date_mask, "last_funding_at_parsed"]
)

print("Records with both funding dates:", funding_date_mask.sum())
print("First funding date after last funding date:", funding_date_errors.sum())

Records with both funding dates: 49438
First funding date after last funding date: 0


In [37]:
founding_funding_mask = (
    df_clean["founded_at_parsed"].notna()
    & df_clean["first_funding_at_parsed"].notna()
)

founding_after_first_funding = (
    df_clean.loc[founding_funding_mask, "founded_at_parsed"]
    > df_clean.loc[founding_funding_mask, "first_funding_at_parsed"]
)

print("Records with both dates:", founding_funding_mask.sum())
print(
    "Founding date after first funding date:",
    founding_after_first_funding.sum()
)

Records with both dates: 38554
Founding date after first funding date: 2745


In [38]:
display(
    df_clean.loc[
        founding_funding_mask
        & (
            df_clean["founded_at_parsed"]
            > df_clean["first_funding_at_parsed"]
        ),
        [
            "name",
            "founded_at",
            "first_funding_at",
            "last_funding_at",
            "founded_year",
            "funding_total_usd"
        ]
    ].head(20)
)

,name,founded_at,first_funding_at,last_funding_at,founded_year,funding_total_usd
2,'Rock' Your Paper,2012-10-26,2012-08-09,2012-08-09,2012.0,"40,000"
20,1000memories,2010-07-01,2010-01-01,2011-02-16,2010.0,"25,35,000"
54,140 Proof,2010-01-11,2009-07-01,2011-04-28,2010.0,"55,00,000"
68,1calendar,2009-01-19,2008-04-04,2008-04-04,2009.0,"40,000"
73,1DayLater,2009-08-26,2009-05-01,2009-10-01,2009.0,"43,811"
77,1EQ,2013-03-08,2012-07-01,2013-12-23,2013.0,"21,00,000"
88,1st Choice Lawn Care,2012-08-01,2012-07-22,2012-07-22,2012.0,-
92,1V1 DRAFT GAEMS,2014-09-19,2014-08-28,2014-08-28,2014.0,-
131,27 Perry,2012-07-25,2012-05-17,2012-05-17,2012.0,"1,18,000"
141,2CODE Online,2008-10-01,2008-01-01,2008-01-01,2008.0,-


In [39]:
total_comparable_records = founding_funding_mask.sum()

inconsistent_records = founding_after_first_funding.sum()

inconsistency_percentage = (
    inconsistent_records / total_comparable_records
) * 100

print("Comparable records:", total_comparable_records)
print("Inconsistent records:", inconsistent_records)
print(f"Inconsistency percentage: {inconsistency_percentage:.2f}%")

Comparable records: 38554
Inconsistent records: 2745
Inconsistency percentage: 7.12%


In [40]:
df_clean["days_founding_after_funding"] = (
    df_clean["founded_at_parsed"]
    - df_clean["first_funding_at_parsed"]
).dt.days

inconsistent_dates = df_clean.loc[
    df_clean["days_founding_after_funding"] > 0,
    "days_founding_after_funding"
]

print("Minimum difference in days:", inconsistent_dates.min())
print("Median difference in days:", inconsistent_dates.median())
print("Average difference in days:", inconsistent_dates.mean())
print("Maximum difference in days:", inconsistent_dates.max())

Minimum difference in days: 1.0
Median difference in days: 117.0
Average difference in days: 1846.696539162113
Maximum difference in days: 735109.0


In [41]:
print("Difference of 1–30 days:",
      inconsistent_dates.between(1, 30).sum())

print("Difference of 31–365 days:",
      inconsistent_dates.between(31, 365).sum())

print("Difference of 1–5 years:",
      inconsistent_dates.between(366, 1825).sum())

print("Difference of more than 5 years:",
      (inconsistent_dates > 1825).sum())

Difference of 1–30 days: 656
Difference of 31–365 days: 1590
Difference of 1–5 years: 436
Difference of more than 5 years: 63


In [42]:
suspicious_date_records = df_clean.loc[
    df_clean["days_founding_after_funding"] > 1825,
    [
        "name",
        "country_code",
        "founded_at",
        "first_funding_at",
        "last_funding_at",
        "founded_year",
        "funding_total_usd",
        "days_founding_after_funding"
    ]
].sort_values(
    by="days_founding_after_funding",
    ascending=False
)

display(suspicious_date_records.head(15))

,name,country_code,founded_at,first_funding_at,last_funding_at,founded_year,funding_total_usd,days_founding_after_funding
31865,PeopleGoal,NaN,2014-01-10,0001-05-14,0001-05-14,2014.0,-,735109.0
29695,Nubank,BRA,2013-01-01,0007-05-13,2014-09-25,2013.0,"1,63,00,000",732545.0
1492,AgFlow,CHE,2012-08-01,0020-06-14,2013-06-01,2012.0,-,727611.0
6661,Buru Buru,ITA,2012-01-01,0019-11-20,2013-04-01,2012.0,-,727605.0
37313,SecureNet Payment Systems,USA,1997-01-01,0011-11-14,2012-07-24,1997.0,"1,80,00,000",725055.0
14524,Exploco,AUS,2014-10-01,0201-01-01,0201-01-01,2014.0,-,662458.0
22250,Jumper Networks,USA,2008-01-05,1960-01-01,1960-01-01,2008.0,-,17536.0
2446,AndrewBurnett.com Ltd,GBR,2008-10-01,1974-01-01,1974-01-01,2008.0,-,12692.0
45355,UTStarcom,CHN,2010-09-01,1996-12-01,2010-02-01,2010.0,"18,05,00,000",5022.0
839,ActionBase,ISR,2007-03-01,1994-07-09,1994-07-09,2007.0,"4,00,000",4618.0


In [43]:
print(
    "Latest first funding date:",
    df_clean["first_funding_at_parsed"].max()
)

print(
    "Latest last funding date:",
    df_clean["last_funding_at_parsed"].max()
)

print(
    "Latest founding date:",
    df_clean["founded_at_parsed"].max()
)

Latest first funding date: 2014-12-31 00:00:00
Latest last funding date: 2015-01-01 00:00:00
Latest founding date: 2014-12-13 00:00:00


In [44]:
df_clean["suspicious_first_funding_date"] = (
    df_clean["first_funding_at_parsed"].dt.year < 2000
)

print(
    "Records with first funding dates before 2000:",
    df_clean["suspicious_first_funding_date"].sum()
)

display(
    df_clean.loc[
        df_clean["suspicious_first_funding_date"],
        [
            "name",
            "country_code",
            "founded_at",
            "first_funding_at",
            "last_funding_at",
            "founded_year",
            "funding_total_usd"
        ]
    ].head(20)
)

Records with first funding dates before 2000: 368


,name,country_code,founded_at,first_funding_at,last_funding_at,founded_year,funding_total_usd
39,12 Labs,NaN,NaN,0026-11-14,0026-11-14,NaN,-
164,3 day Blinds,USA,1984-01-01,1998-03-13,1998-03-13,1984.0,-
266,404 Found!,NaN,1997-01-01,1999-07-01,2000-07-01,1997.0,"62,46,800"
482,A+ Network,USA,1989-01-01,1994-04-14,1994-04-14,1989.0,-
574,ABO Data,USA,1979-01-01,1979-01-01,1979-01-01,1979.0,"10,00,000"
617,Academic Management Services,USA,1972-01-01,1993-11-30,1993-11-30,1972.0,-
839,ActionBase,ISR,2007-03-01,1994-07-09,1994-07-09,2007.0,"4,00,000"
870,ACTIVE Network,USA,1999-08-01,1999-09-01,2009-12-11,1999.0,"21,42,08,700"
875,Active Voice Corporation,USA,1983-01-01,1990-10-31,1990-10-31,1983.0,-
1002,Addashop,NaN,NaN,1999-01-01,1999-01-01,NaN,"30,00,000"


In [45]:
early_funding_years = (
    df_clean.loc[
        df_clean["suspicious_first_funding_date"],
        "first_funding_at_parsed"
    ]
    .dt.year
    .value_counts()
    .sort_index()
)

display(early_funding_years)

first_funding_at_parsed
1         3
7         1
11        1
19        1
20        1
26        1
29        1
201       1
1921      1
1960      3
1973      1
1974      2
1979      1
1982      3
1983      1
1984      4
1985      6
1986      4
1987      6
1988      2
1989      4
1990     14
1991      9
1992     12
1993     15
1994     13
1995     19
1996     30
1997     24
1998     53
1999    131
Name: count, dtype: int64

In [46]:
df_clean["suspicious_funding_date"] = (
    (df_clean["first_funding_at_parsed"].dt.year < 2000)
    & (df_clean["founded_year"] >= 2000)
)

print("Suspicious funding-date records:",
      df_clean["suspicious_funding_date"].sum())

Suspicious funding-date records: 29


In [47]:
important_columns = [
    "name",
    "market",
    "category_list",
    "country_code",
    "status",
    "funding_total_numeric",
    "funding_rounds",
    "founded_year",
    "first_funding_at_parsed",
    "last_funding_at_parsed"
]

missing_summary = pd.DataFrame({
    "Missing Count": df_clean[important_columns].isna().sum(),
    "Missing Percentage": (
        df_clean[important_columns].isna().mean() * 100
    ).round(2)
})

display(missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
))

,Missing Count,Missing Percentage
founded_year,10956,22.16
funding_total_numeric,8531,17.26
country_code,5273,10.67
market,3968,8.03
category_list,3961,8.01
status,1314,2.66
name,1,0.00
funding_rounds,0,0.00
first_funding_at_parsed,0,0.00
last_funding_at_parsed,0,0.00


In [48]:
display(
    df_clean.loc[
        df_clean["name"].isna(),
        ["permalink", "name", "country_code", "market",
         "funding_total_numeric", "status"]
    ]
)

,permalink,name,country_code,market,funding_total_numeric,status
28221,/organization/tell-it-in,NaN,NaN,Startups,25000,closed


In [49]:
duplicate_companies = df_clean[
    df_clean["permalink"].duplicated(keep=False)
].sort_values("permalink")

print("Records with repeated company identifiers:",
      len(duplicate_companies))

display(duplicate_companies[
    ["permalink", "name", "funding_total_numeric",
     "funding_rounds", "first_funding_at",
     "last_funding_at", "country_code", "category_list"]
])

Records with repeated company identifiers: 4


,permalink,name,funding_total_numeric,funding_rounds,first_funding_at,last_funding_at,country_code,category_list
33939,/organization/prysm,Prysm,293080123,1,2014-11-25,2014-11-25,NaN,NaN
33940,/organization/prysm,Prysm,293080123,3,2010-01-05,2011-10-27,USA,|Displays|Hardware + Software|
44033,/organization/treasure-valley-urology-services,Treasure Valley Urology Services,332194,4,2009-09-09,2014-05-05,USA,|Biotechnology|
44034,/organization/treasure-valley-urology-services,Treasure Valley Urology Services,332194,1,2014-07-18,2014-07-18,USA,NaN


In [50]:
# Remove temporary investigation columns
temporary_columns = [
    "funding_components_sum",
    "funding_difference",
    "days_founding_after_funding"
]

df_clean = df_clean.drop(
    columns=temporary_columns,
    errors="ignore"
)

# Verify the final dataset
print("Final dataset shape:", df_clean.shape)
print("Exact duplicate rows:", df_clean.duplicated().sum())
print(
    "Repeated company identifiers:",
    df_clean["permalink"].duplicated().sum()
)

display(df_clean.head())

Final dataset shape: (49438, 45)
Exact duplicate rows: 0
Repeated company identifiers: 2


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H,funding_total_numeric,founded_at_parsed,first_funding_at_parsed,last_funding_at_parsed,suspicious_first_funding_date,suspicious_funding_date
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,New York,1,2012-06-01,2012-06,2012-Q2,2012.0,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1750000,2012-06-01,2012-06-30,2012-06-30,False,False
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,Los Angeles,2,NaN,NaN,NaN,NaN,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4000000,NaT,2010-06-04,2010-09-23,False,False
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,Tallinn,1,2012-10-26,2012-10,2012-Q4,2012.0,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40000,2012-10-26,2012-08-09,2012-08-09,False,False
3,/organization/in-touch-network,(In)Touch Network,http://www.InTouchNetwork.com,|Electronics|Guides|Coffee|Restaurants|Music|i...,Electronics,"15,00,000",operating,GBR,NaN,London,London,1,2011-04-01,2011-04,2011-Q2,2011.0,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1500000,2011-04-01,2011-04-01,2011-04-01,False,False
4,/organization/r-ranch-and-mine,-R- Ranch and Mine,NaN,|Tourism|Entertainment|Games|,Tourism,"60,000",operating,USA,TX,Dallas,Fort Worth,2,2014-01-01,2014-01,2014-Q1,2014.0,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60000,2014-01-01,2014-08-17,2014-09-26,False,False


In [51]:
import os

output_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed"

os.makedirs(output_path, exist_ok=True)

df_clean.to_csv(
    os.path.join(output_path, "global_startup_cleaned.csv"),
    index=False
)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [2]:
import pandas as pd
import os

input_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed\global_startup_cleaned.csv"

df_clean = pd.read_csv(input_path)

sql_columns = [
    "permalink",
    "name",
    "market",
    "category_list",
    "country_code",
    "state_code",
    "region",
    "city",
    "status",
    "founded_year",
    "funding_rounds",
    "funding_total_numeric",
    "first_funding_at_parsed",
    "last_funding_at_parsed",
    "suspicious_funding_date"
]

df_sql = df_clean[sql_columns].copy()

# Convert numeric columns safely
for col in ["founded_year", "funding_rounds", "funding_total_numeric"]:
    df_sql[col] = pd.to_numeric(df_sql[col], errors="coerce")

# Convert Boolean flags to 0 and 1
df_sql["suspicious_funding_date"] = (
    df_sql["suspicious_funding_date"]
    .map({True: 1, False: 0, "True": 1, "False": 0})
)

# Convert date columns to proper dates
date_columns = [
    "first_funding_at_parsed",
    "last_funding_at_parsed"
]

for col in date_columns:
    df_sql[col] = pd.to_datetime(df_sql[col], errors="coerce").dt.strftime("%Y-%m-%d")

# Export without writing NaN or NaT as text
output_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed"

os.makedirs(output_path, exist_ok=True)

df_sql.to_csv(
    os.path.join(output_path, "global_startup_sql.csv"),
    index=False,
    na_rep=""
)

print("Rows:", len(df_sql))
print("Columns:", len(df_sql.columns))
print("Missing founded_year:", df_sql["founded_year"].isna().sum())
print("Missing funding_total_numeric:", df_sql["funding_total_numeric"].isna().sum())
print("CSV exported successfully!")

Rows: 49438
Columns: 15
Missing founded_year: 10956
Missing funding_total_numeric: 8531
CSV exported successfully!


In [1]:
%pip install mysql-connector-python

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.2 MB 953.1 kB/s eta 0:00:19
   - -------------------------------------- 0.8/18.2 MB 1.1 MB/s eta 0:00:17
   -- ------------------------------------- 1.0/18.2 MB 1.2 MB/s eta 0:00:15
   --- ------------------------------------ 1.6/18.2 MB 1.2 MB/s eta 0:00:15
   ---- ----------------------------------- 1.8/18.2 MB 1.2 MB/s eta 0:00:14
   ----- ---------------------------------- 2.4/18.2 MB 1.4 MB/s eta 0:00:12
   ------ --------------------------------- 2.9/18.2 MB 1.5 MB/s eta 0:00:11
   ------- -------------------------------- 3.4/18.2 MB 1.6 MB/s eta 0:00:10
   -------- ------------------------------- 3.9/18.2 MB 1.7 MB/s eta 0:00:09
   ---------- -----------

In [2]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Bhaskar@21",
    database="global_startup_analysis"
)

cursor = connection.cursor()

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [3]:
import pandas as pd

csv_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed\global_startup_sql.csv"

df_sql = pd.read_csv(csv_path)

print("Rows:", len(df_sql))
print("Columns:", len(df_sql.columns))
print(df_sql.columns.tolist())

Rows: 49438
Columns: 15
['permalink', 'name', 'market', 'category_list', 'country_code', 'state_code', 'region', 'city', 'status', 'founded_year', 'funding_rounds', 'funding_total_numeric', 'first_funding_at_parsed', 'last_funding_at_parsed', 'suspicious_funding_date']


In [4]:
create_table_query = """
CREATE TABLE IF NOT EXISTS startups (
    permalink VARCHAR(255),
    name VARCHAR(255),
    market VARCHAR(255),
    category_list TEXT,
    country_code VARCHAR(10),
    state_code VARCHAR(20),
    region VARCHAR(255),
    city VARCHAR(255),
    status VARCHAR(50),
    founded_year SMALLINT,
    funding_rounds INT,
    funding_total_numeric BIGINT,
    first_funding_at_parsed DATE,
    last_funding_at_parsed DATE,
    suspicious_funding_date TINYINT(1)
)
"""

cursor.execute(create_table_query)
connection.commit()

print("Table created successfully!")

Table created successfully!


In [1]:
import pandas as pd
import mysql.connector

# CSV path
csv_path = r"C:\Users\bhask\OneDrive\Desktop\My data\Projects\Data Analytics Projects\Global-Startup-Analytics\data\processed\global_startup_sql.csv"

# Load all CSV fields as strings; preserve empty fields for explicit NULL handling
df_sql = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

# Convert empty strings to None (SQL NULL)
df_sql = df_sql.where(df_sql != "", None)

# Connect to MySQL if you do not already have an active connection
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Bhaskar@21",
    database="global_startup_analysis"
)
cursor = connection.cursor()

# Prevent accidentally duplicating previously imported rows
cursor.execute("SELECT COUNT(*) FROM startups")
existing_rows = cursor.fetchone()[0]

if existing_rows != 0:
    raise ValueError(
        f"The startups table already contains {existing_rows} rows. "
        "Import cancelled to avoid duplicate records."
    )

# Build INSERT statement from the CSV columns
columns = df_sql.columns.tolist()

insert_query = f"""
INSERT INTO startups ({', '.join(f'`{col}`' for col in columns)})
VALUES ({', '.join(['%s'] * len(columns))})
"""

# Insert in batches
batch_size = 1000

for start in range(0, len(df_sql), batch_size):
    batch = df_sql.iloc[start:start + batch_size]
    values = [
        tuple(None if pd.isna(value) or value == "" else str(value)
              for value in row)
        for row in batch.itertuples(index=False, name=None)
    ]

    cursor.executemany(insert_query, values)
    connection.commit()

    print(f"Imported {min(start + batch_size, len(df_sql)):,} / {len(df_sql):,} rows")

print("Import completed successfully!")

cursor.execute("SELECT COUNT(*) FROM startups")
print("Total rows in MySQL:", cursor.fetchone()[0])

Imported 1,000 / 49,438 rows
Imported 2,000 / 49,438 rows
Imported 3,000 / 49,438 rows
Imported 4,000 / 49,438 rows
Imported 5,000 / 49,438 rows
Imported 6,000 / 49,438 rows
Imported 7,000 / 49,438 rows
Imported 8,000 / 49,438 rows
Imported 9,000 / 49,438 rows
Imported 10,000 / 49,438 rows
Imported 11,000 / 49,438 rows
Imported 12,000 / 49,438 rows
Imported 13,000 / 49,438 rows
Imported 14,000 / 49,438 rows
Imported 15,000 / 49,438 rows
Imported 16,000 / 49,438 rows
Imported 17,000 / 49,438 rows
Imported 18,000 / 49,438 rows
Imported 19,000 / 49,438 rows
Imported 20,000 / 49,438 rows
Imported 21,000 / 49,438 rows
Imported 22,000 / 49,438 rows
Imported 23,000 / 49,438 rows
Imported 24,000 / 49,438 rows
Imported 25,000 / 49,438 rows
Imported 26,000 / 49,438 rows
Imported 27,000 / 49,438 rows
Imported 28,000 / 49,438 rows
Imported 29,000 / 49,438 rows
Imported 30,000 / 49,438 rows
Imported 31,000 / 49,438 rows
Imported 32,000 / 49,438 rows
Imported 33,000 / 49,438 rows
Imported 34,000 / 4